# SOEN-321: Secure Messaging Demo

This notebook demonstrates:
- Encryption and signing (Alice)
- Transmission via an untrusted server
- Verification and decryption (Bob)
- Tampering detection

Before running this notebook:
1. Install Project
    ```bash
    uv pip install -e .
    ```
2. Start API
    ```bash
    uv run uvicorn api.main:app --reload
    ```

In [15]:
import sys
from pathlib import Path

import requests

sys.path.append(str(Path().resolve().parents[0] / "src"))

from api.crypto import encrypt_and_sign, verify_and_decrypt, tamper_ciphertext, generate_demo_keys

In [16]:
SERVER = "http://127.0.0.1:8000"
MSG_ID = "demo1"

## Generate Public/Private Keys

In [17]:
generate_demo_keys()
print("Demo keys generated in ./keys")

Demo keys generated in ./keys


## Alice

In [18]:
plaintext = b"Hello Bob. This message is encrypted and the server cannot read it."

package = encrypt_and_sign(plaintext)
print("Alice is sending encrypted package:")
display(package)

response = requests.post(f"{SERVER}/messages/{MSG_ID}", json=package, timeout=10)

print("\nServer response:")
display(response.status_code, response.json())

Alice is sending encrypted package:


{'algorithms': {'data': 'AES-256-GCM',
  'key_wrap': 'RSA-2048-OAEP-SHA256',
  'signature': 'Ed25519'},
 'nonce': 'ppHB6fCKZKv49pFu',
 'enc_key': 'I7IcozcUMtA7pQNPpyWH5YVcZ18dSA9yuO0GP0raPWr4B7vVmjFN9hQU51/8aCuKIc6ZM1+s/Nj+baq0tBLhYlH7wgRcxorqv7zYxvT3S+Cz9+WPeBdwYm53qtPWwF/3tC/WYF9drS0NO7SHk7MqS2asUIbHcs4EKmveXrF7YEOyA5UmA3Zuc7Ck3EqPJWs0FhbZWxEuBDwSedQgVSE5DJ0QpPJFsYiX5gLokTmvwSTsvldg3OmNVnkwxTbA2yaqloTjAlyQwjGtc0/4bZJHw29mFKNWMItVYSXEbw4LLcODwnzL71Zwd3m4+jb7B64C0VnUjSLmiiA+HRExDiFk6A==',
 'ciphertext': '20DFK51o07Hl5ki1xGiKimqzjV0hTgBlbzixJ1BDpGgQagKh8ga4zI77MFI3GxVOD6KLkmRgMHaeLGdqz61j/3dLSaV26XDf5kjCqM86v961xI8=',
 'signature': '9fhoZnyvT5+Jyt5zyMEaiiCbeX8aLlc8ycYN6Y4j3d3WyQo1xHj+s2bHpy0S5kIjAjr0Y/ZGx+zoFM4VXZDHBw=='}


Server response:


200

{'status': 'stored', 'message_id': 'demo1'}

In [19]:
response = requests.get(f"{SERVER}/messages/{MSG_ID}", timeout=10)
package = response.json()

print("Stored package (what the server sees):")
display(package)

Stored package (what the server sees):


{'nonce': 'ppHB6fCKZKv49pFu',
 'enc_key': 'I7IcozcUMtA7pQNPpyWH5YVcZ18dSA9yuO0GP0raPWr4B7vVmjFN9hQU51/8aCuKIc6ZM1+s/Nj+baq0tBLhYlH7wgRcxorqv7zYxvT3S+Cz9+WPeBdwYm53qtPWwF/3tC/WYF9drS0NO7SHk7MqS2asUIbHcs4EKmveXrF7YEOyA5UmA3Zuc7Ck3EqPJWs0FhbZWxEuBDwSedQgVSE5DJ0QpPJFsYiX5gLokTmvwSTsvldg3OmNVnkwxTbA2yaqloTjAlyQwjGtc0/4bZJHw29mFKNWMItVYSXEbw4LLcODwnzL71Zwd3m4+jb7B64C0VnUjSLmiiA+HRExDiFk6A==',
 'ciphertext': '20DFK51o07Hl5ki1xGiKimqzjV0hTgBlbzixJ1BDpGgQagKh8ga4zI77MFI3GxVOD6KLkmRgMHaeLGdqz61j/3dLSaV26XDf5kjCqM86v961xI8=',
 'signature': '9fhoZnyvT5+Jyt5zyMEaiiCbeX8aLlc8ycYN6Y4j3d3WyQo1xHj+s2bHpy0S5kIjAjr0Y/ZGx+zoFM4VXZDHBw==',
 'algorithms': {'data': 'AES-256-GCM',
  'key_wrap': 'RSA-2048-OAEP-SHA256',
  'signature': 'Ed25519'}}

## Bob

In [20]:
plaintext = verify_and_decrypt(package)

print("Bob decrypted the message:")
display(plaintext.decode("utf-8"))

Bob decrypted the message:


'Hello Bob. This message is encrypted and the server cannot read it.'

## Tamper

In [21]:
bad_package = tamper_ciphertext(package)

print("Tampered package created.")

Tampered package created.


## Bob: Reads tampered message

In [22]:
try:
    verify_and_decrypt(bad_package)
    print("Unexpected: tampered message decrypted successfully")
except Exception as e:
    print("Tampering detected!")
    print("Error:", type(e).__name__, str(e))

Tampering detected!
Error: InvalidSignature 
